# Lab 3 — Environments, Project Structure & Reliable Code
**ARTI 303 — Programming for AI**

**Name:** _[your name]_  **Student ID:** _[your ID]_  **Section:** _[your section]_

---

### The problem this lab solves

You write code. It works on your laptop. You send it to a classmate, or
open it on a lab machine, or come back to it in two months — and it
breaks.

Nothing about your code changed. What changed was everything *around*
it: which Python, which packages, which versions, which folder things
sat in.

This lab is about making that stop happening. By the end you'll have a
project that runs on a machine that has never seen it before — which is
the real test of whether code is finished.

### What you'll build

A small, properly structured project:

```
your-repo/
├── src/           reusable code (.py modules)
├── tests/         automated tests
├── data/          input files
├── logs/          output from your program
├── requirements.txt
├── .gitignore
└── README.md
```

### How to use this notebook

Read, run, then do the exercise. Self-check cells tell you immediately
whether you got it right.

Some commands in this lab run in the **terminal**, not in this notebook.
Those are clearly marked. Everything in a code cell runs here.

**Work through in order** — later parts depend on files created earlier.

---
# Part 1 · Why environments break

By default, `pip install` puts packages in one shared place for your
whole computer. Every project uses the same pile.

That works until two projects disagree:

| | needs |
|---|---|
| Project A | `numpy 1.24` |
| Project B | `numpy 2.1` |

One shared pile means one version wins and the other project breaks.
You can't fix this by being careful — it's structural.

### The fix: one isolated environment per project

A **virtual environment** is a private folder of packages belonging to
one project only. Project A gets its own `numpy`, Project B gets its
own, and neither can break the other.

Let's see where your Python currently is.

In [1]:
import sys

print("Python executable:")
print(" ", sys.executable)
print()
print("Version:", sys.version.split()[0])
print()

in_venv = sys.prefix != sys.base_prefix
print("Inside a virtual environment?", in_venv)

Python executable:
  /opt/homebrew/opt/python@3.11/bin/python3.11

Version: 3.11.5

Inside a virtual environment? False


`sys.prefix` is where the *currently running* Python lives.
`sys.base_prefix` is where the system Python lives. When they differ,
you're inside a virtual environment.

Right now they're probably the same — that's expected. You'll change it
in Part 2.

---
# Part 2 · Creating a virtual environment

**These commands run in the terminal**, not in this notebook.
Open VS Code's terminal (`` Ctrl+` ``) in your repository folder.

### Create it

```bash
python -m venv .venv
```

That makes a folder called `.venv` holding a private copy of Python.
The name `.venv` is a convention — the leading dot marks it as
infrastructure, not your work.

### Activate it

**macOS / Linux / Git Bash:**
```bash
source .venv/bin/activate
```

**Windows PowerShell:**
```bash
.venv\Scripts\Activate.ps1
```

Your prompt changes to show `(.venv)` at the start. That's how you know
it worked.

### Verify it

```bash
python -c "import sys; print(sys.prefix != sys.base_prefix)"
```

Should print `True`.

> **Activation is per-terminal, not permanent.** Every new terminal
> starts deactivated. If a command suddenly can't find a package you
> definitely installed, check for `(.venv)` in your prompt first — this
> is the single most common confusion with virtual environments.

### Point this notebook at it

Restart the kernel, then select the `.venv` interpreter:
`Ctrl+Shift+P` → **Python: Select Interpreter** → pick the one showing
`.venv`. Then re-run the cell below.

### Exercise 2.1
Run the check below. It reports your status rather than failing — but
you should see `True` before moving on.

In [ ]:
import sys

in_venv = sys.prefix != sys.base_prefix

if in_venv:
    print("Inside a virtual environment.")
else:
    print("NOT inside a virtual environment.")
    print("Create and activate .venv, then switch this notebook's interpreter to it.")

print()
print("prefix:      ", sys.prefix)
print("base_prefix: ", sys.base_prefix)

> If you're working in Google Colab, you're already isolated — Colab
> gives every session a fresh environment. The check above will say
> `False`, and that's fine. Read Part 2 anyway; you need it for any work
> on your own machine.

---
# Part 3 · Recording your dependencies

An environment nobody can rebuild is no better than no environment.
`requirements.txt` is the recipe: a plain text file listing what your
project needs.

### Installing

With `.venv` active:

```bash
python -m pip install pytest
```

This installs *into the environment*, not system-wide.

### Recording what's installed

```bash
python -m pip freeze > requirements.txt
```

`pip freeze` lists every installed package with its exact version, and
`>` writes that into the file.

### Pinning: `==` vs `>=`

| Written as | Means | Result |
|---|---|---|
| `pandas==2.1.0` | exactly this version | reproducible |
| `pandas>=2.1.0` | this or anything newer | may change without warning |
| `pandas` | anything at all | different on every machine |

For coursework, **pin with `==`**. A project that says `pandas` will
install whatever is current, and "current" changes. Your results should
not depend on which day someone ran your code.

### Reading it back

Let's write a small file and parse it — a chance to use the strings,
lists, and dicts you already know.

In [ ]:
# Create a sample requirements file to work with
sample_requirements = """# Core dependencies
pytest==9.1.1
numpy==2.1.0

# Optional
matplotlib>=3.8
"""

with open("sample_requirements.txt", "w") as f:
    f.write(sample_requirements)

print(open("sample_requirements.txt").read())

That's your first **file write**. `open(path, "w")` opens for writing,
and the `with` block closes the file automatically when it ends — even
if an error happens inside. Always use `with` for files.

### Exercise 3.1
Write `parse_requirements(path)` that reads a requirements file and
returns a **dictionary** mapping package name to pinned version, using
only lines pinned with `==`.

- Skip blank lines
- Skip comment lines (starting with `#`)
- Skip lines that aren't pinned with `==`

For the file above it should return
`{"pytest": "9.1.1", "numpy": "2.1.0"}` — note `matplotlib` is excluded
because it uses `>=`.

Useful: `line.strip()` removes surrounding whitespace,
`line.startswith("#")` tests for comments, `"a==b".split("==")` gives
`["a", "b"]`.

In [ ]:
def parse_requirements(path):
    """Return {package: version} for lines pinned with ==."""
    pinned = {}
    with open(path) as f:
        for line in f:
            # TODO: strip the line, skip blanks and comments,
            #       keep only lines containing "==", split and store
            pass
    return pinned


result = parse_requirements("sample_requirements.txt")
print(result)

In [ ]:
assert result == {"pytest": "9.1.1", "numpy": "2.1.0"}, \
    f"expected {{'pytest': '9.1.1', 'numpy': '2.1.0'}}, got {result}"
print("Exercise 3.1 correct.")

---
# Part 4 · Project structure

Loose files in one folder stop working once a project has more than a
handful. A standard layout means anyone — including you in two months —
knows where to look.

| Folder | Holds |
|---|---|
| `src/` | reusable `.py` modules |
| `tests/` | automated tests |
| `data/` | input files |
| `logs/` | program output |
| `notebooks/` | exploration and analysis |

Let's build it.

In [ ]:
import os

for folder in ["src", "tests", "data", "logs"]:
    os.makedirs(folder, exist_ok=True)
    print("ready:", folder)

`exist_ok=True` means "don't complain if it's already there" — so this
cell is safe to re-run.

### Putting a module in `src/`

Below is a grading module for our 5.00 scale. `%%writefile` saves the
cell to a file instead of running it.

In [ ]:
%%writefile src/grade_utils.py
"""Grade helpers for the 5.00 GPA scale."""

DEANS_LIST_MIN = 4.50


def letter_grade(gpa):
    """Convert a GPA out of 5.00 into a letter grade."""
    if gpa >= 4.50:
        return "A"
    elif gpa >= 3.50:
        return "B"
    elif gpa >= 2.50:
        return "C"
    elif gpa >= 1.50:
        return "D"
    else:
        return "F"


def is_dean_list(gpa):
    """Return True if this GPA meets the Dean's List threshold."""
    return gpa >= DEANS_LIST_MIN


### Importing from `src/`

Python only looks for modules in certain places, and `src/` isn't one of
them by default. Adding it to `sys.path` tells Python to look there
too.

In [ ]:
import sys

if "src" not in sys.path:
    sys.path.insert(0, "src")

import grade_utils

print(grade_utils.letter_grade(4.85))
print(grade_utils.is_dean_list(4.60))
print("threshold:", grade_utils.DEANS_LIST_MIN)

> **Remember:** Python loads a module once per session. Edit
> `src/grade_utils.py` and this notebook keeps the old version until you
> **Restart Kernel and Run All**. When an edit to a `.py` file seems to
> do nothing, this is almost always why.

> `sys.path.insert` is the simple approach and it's fine at this size.
> Larger projects install themselves as a package instead, which removes
> the need for path juggling.

### Exercise 4.1
Create `src/text_utils.py` with one function, `clean_name(raw)`, that
tidies a messy name string:

- Remove surrounding whitespace
- Convert to **title case** (`"sara ali"` → `"Sara Ali"`)
- Collapse repeated inner spaces into one (`"Sara   Ali"` → `"Sara Ali"`)

Hint: `" ".join(raw.split())` collapses all whitespace, because
`.split()` with no argument splits on any run of whitespace.

Give the module and function docstrings.

In [ ]:
%%writefile src/text_utils.py
"""TODO: describe this module."""


def clean_name(raw):
    """TODO: describe this function."""
    # TODO: collapse whitespace, then title-case
    pass


In [ ]:
import importlib
import text_utils
importlib.reload(text_utils)   # picks up the file you just wrote

print(repr(text_utils.clean_name("  sara   ali  ")))
print(repr(text_utils.clean_name("FAISAL ALHARBI")))

In [ ]:
assert text_utils.clean_name("  sara   ali  ") == "Sara Ali", \
    f'got {text_utils.clean_name("  sara   ali  ")!r}'
assert text_utils.clean_name("FAISAL ALHARBI") == "Faisal Alharbi", \
    f'got {text_utils.clean_name("FAISAL ALHARBI")!r}'
assert text_utils.clean_name("lama") == "Lama"
print("Exercise 4.1 correct.")

---
# Part 5 · Reading data from files

Real data arrives in files, not typed into your code. Let's make one.

In [ ]:
csv_text = """name,gpa
Sara Alotaibi,4.60
Faisal Alharbi,3.90
Lama Alqahtani,4.85
Omar Alshehri,4.40
Noura Alzahrani,2.95
"""

with open("data/students.csv", "w") as f:
    f.write(csv_text)

print("written to data/students.csv")

### Reading it back

Three common ways to read a file:

In [ ]:
# 1. The whole thing as one string
with open("data/students.csv") as f:
    content = f.read()
print("read():", repr(content[:40]), "...")
print()

# 2. As a list of lines
with open("data/students.csv") as f:
    lines = f.readlines()
print("readlines():", lines[:2])
print()

# 3. Line by line  <- usually what you want
with open("data/students.csv") as f:
    for line in f:
        print(repr(line.strip()))

Option 3 is the default choice: it reads one line at a time instead of
loading everything into memory, so it works the same on a 10-line file
and a 10-million-line one.

Note each line ends with `\n` — `.strip()` removes it.

### Turning lines into data

Combine file reading with the strings, lists and dicts you know.

In [ ]:
def load_students(path):
    """Read a name,gpa CSV into a list of dictionaries."""
    students = []
    with open(path) as f:
        header = f.readline()          # skip the header row
        for line in f:
            line = line.strip()
            if not line:
                continue
            name, gpa = line.split(",")
            students.append({"name": name, "gpa": float(gpa)})
    return students


roster = load_students("data/students.csv")
for s in roster:
    print(s)

`float(gpa)` matters: everything read from a file is a **string**.
`"4.60"` is text; `4.60` is a number. Comparing `"4.60" >= 4.50` raises
an error, and forgetting the conversion is one of the most common file-
reading bugs.

### Exercise 5.1
Write `dean_list_from_file(path)` that loads the file and returns a
**list of names** of students meeting the Dean's List threshold.

Reuse `load_students` and `grade_utils.is_dean_list` — don't rewrite
either.

In [ ]:
def dean_list_from_file(path):
    """Return the names of students on the Dean's List."""
    # TODO: load the file, then filter using grade_utils.is_dean_list
    pass


names = dean_list_from_file("data/students.csv")
print(names)

In [ ]:
assert names == ["Sara Alotaibi", "Lama Alqahtani"], f"got {names}"
print("Exercise 5.1 correct.")

### Writing results out

In [ ]:
with open("logs/dean_list.txt", "w") as f:
    for name in names:
        f.write(name + "\n")

print(open("logs/dean_list.txt").read())

> `"w"` overwrites the file completely. `"a"` appends to it. Using
> `"w"` when you meant `"a"` silently destroys the previous contents —
> worth checking twice.

---
# Part 6 · Handling errors

`load_students` works — until the data isn't perfect. And real data is
never perfect.

In [ ]:
# A file with problems in it
messy = """name,gpa
Sara Alotaibi,4.60
Faisal Alharbi,N/A
Lama Alqahtani,4.85
Omar Alshehri,
Noura Alzahrani,2.95
"""

with open("data/students_messy.csv", "w") as f:
    f.write(messy)

# What happens with our current function?
try:
    load_students("data/students_messy.csv")
except ValueError as e:
    print("ValueError:", e)

One bad row and the whole load fails — losing four perfectly good
records because of one broken one.

### try / except

```python
try:
    # code that might fail
except SomeError as e:
    # what to do if it does
```

Catch the **specific** error you expect. A bare `except:` swallows
everything, including bugs you'd rather find out about.

Errors you'll meet constantly:

| Error | Cause |
|---|---|
| `FileNotFoundError` | the path doesn't exist |
| `ValueError` | right type, wrong content — `float("N/A")` |
| `KeyError` | dictionary key that isn't there |
| `IndexError` | list position out of range |
| `TypeError` | wrong type entirely — `"4.6" + 1` |

### Exercise 6.1
Write `load_students_safe(path)` that:

- Returns `[]` if the file doesn't exist — no crash
- Skips rows whose GPA can't be converted, keeping the good ones
- Returns the list of valid students

For the messy file it should return the **3 valid** students
(Sara, Lama, Noura).

In [ ]:
def load_students_safe(path):
    """Load students, skipping bad rows. Return [] if the file is missing."""
    students = []
    # TODO: wrap the file opening in try/except FileNotFoundError
    # TODO: inside the loop, wrap float() in try/except ValueError and skip bad rows
    return students


good = load_students_safe("data/students_messy.csv")
print("loaded:", len(good))
for s in good:
    print(" ", s)

print()
print("missing file ->", load_students_safe("data/does_not_exist.csv"))

In [ ]:
assert len(good) == 3, f"expected 3 valid students, got {len(good)}"
assert [s["name"] for s in good] == ["Sara Alotaibi", "Lama Alqahtani", "Noura Alzahrani"], \
    f'got {[s["name"] for s in good]}'
assert load_students_safe("data/does_not_exist.csv") == [], \
    "a missing file should return [] rather than raising"
print("Exercise 6.1 correct.")

> Silently skipping bad rows is its own problem — you now have no idea
> anything went wrong. That's what Part 7 fixes.

---
# Part 7 · Logging

`print()` is fine while you're writing code. It stops being fine when
the program runs unattended, because nothing is kept.

Logging gives you: severity levels, timestamps, and a permanent file.

| Level | Use for |
|---|---|
| `DEBUG` | detailed tracing while developing |
| `INFO` | normal progress — "loaded 5 records" |
| `WARNING` | something odd but survivable — "skipped a bad row" |
| `ERROR` | something failed |
| `CRITICAL` | the program cannot continue |

### Setting up a logger

In [ ]:
import logging

logger = logging.getLogger("arti303")
logger.setLevel(logging.INFO)
logger.handlers.clear()        # so re-running this cell doesn't duplicate output

# to a file
file_handler = logging.FileHandler("logs/app.log", mode="w")
file_handler.setFormatter(
    logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s")
)
logger.addHandler(file_handler)

# and to the screen
console = logging.StreamHandler()
console.setFormatter(logging.Formatter("%(levelname)-8s | %(message)s"))
logger.addHandler(console)

logger.info("Logger ready")
logger.warning("This is what a warning looks like")

### Exercise 7.1
Write `load_students_logged(path)` — same behaviour as
`load_students_safe`, but it also logs:

- `logger.error(...)` if the file is missing
- `logger.warning(...)` for each row skipped
- `logger.info(...)` at the end, reporting how many were loaded

Include the actual values in your messages — `f"Loaded {n} students"` is
useful, `"Done"` is not.

In [ ]:
def load_students_logged(path):
    """Load students with logging for missing files and bad rows."""
    students = []
    # TODO: same structure as load_students_safe, plus logger calls
    return students


result = load_students_logged("data/students_messy.csv")
print()
print("returned:", len(result), "students")

In [ ]:
assert len(result) == 3, f"expected 3 students, got {len(result)}"
assert load_students_logged("data/nope.csv") == [], "missing file should return []"

log_contents = open("logs/app.log").read()
assert "WARNING" in log_contents, "bad rows should produce WARNING entries"
assert "ERROR" in log_contents, "a missing file should produce an ERROR entry"
assert "INFO" in log_contents, "a successful load should produce an INFO entry"
print("Exercise 7.1 correct.")

In [ ]:
print(open("logs/app.log").read())

That file persists after your program ends. When something goes wrong
at 3am in a job nobody was watching, this is what you read.

> `logs/` holds generated output, not source. It belongs in
> `.gitignore` — more on that in Part 9.

---
# Part 8 · PEP 8 — the shared style

**PEP 8** is Python's official style guide. Following it isn't about
being tidy; it's about every Python developer being able to read your
code without adjusting.

| Thing | Convention | Example |
|---|---|---|
| Variables, functions | `snake_case` | `student_count`, `load_data()` |
| Constants | `UPPER_CASE` | `DEANS_LIST_MIN` |
| Classes | `CapWords` | `StudentRecord` |
| Indentation | 4 spaces | never tabs |
| Line length | ≤ 79 chars | wrap longer lines |
| Blank lines | 2 between top-level functions | |
| Imports | one per line, at the top | |

Compare the same function written both ways.

In [ ]:
# Not PEP 8 — works, but fights the reader
def CalcAvg(L):
    T=0
    for i in L: T=T+i
    return T/len(L)

print(CalcAvg([4.6, 3.9, 4.85]))

In [ ]:
# PEP 8
def calculate_average(values):
    """Return the average of a list of numbers."""
    total = 0
    for value in values:
        total += value
    return total / len(values)


print(calculate_average([4.6, 3.9, 4.85]))

Both produce the same number. The second tells you what it does, uses
names another developer expects, and has somewhere obvious to add an
empty-list check.

> VS Code can do this for you. Install the **Black Formatter**
> extension, then enable *Format on Save* — your files get formatted to
> a consistent style every time you press save, and you stop thinking
> about it.

### Exercise 8.1
Rewrite the badly-styled function below to follow PEP 8:

- `snake_case` function and variable names
- 4-space indentation, one statement per line
- A docstring
- Handle the empty-list case by returning `0`

Keep the behaviour: it counts how many GPAs are on the Dean's List.

In [ ]:
# Rewrite this:
#
# def CountDL(L):
#     C=0
#     for x in L:
#         if x>=4.5: C=C+1
#     return C

# TODO: your PEP 8 version, named count_dean_list
def count_dean_list(gpas):
    pass


print(count_dean_list([4.6, 3.9, 4.85, 4.4]))
print(count_dean_list([]))

In [ ]:
assert count_dean_list([4.6, 3.9, 4.85, 4.4]) == 2, "expected 2"
assert count_dean_list([]) == 0, "an empty list should return 0"
assert count_dean_list([5.0, 4.5]) == 2, "4.50 is on the boundary and counts"
assert count_dean_list([1.0, 2.0]) == 0, "expected 0"
print("Exercise 8.1 correct.")

---
# Part 9 · What Git should never see

Your repository should contain your **work**, not everything that
happens to be in the folder.

| Never commit | Why |
|---|---|
| `.venv/` | thousands of files, and it's machine-specific — `requirements.txt` replaces it |
| `__pycache__/` | compiled files, regenerated automatically |
| `logs/` | program output, not source |
| `kaggle.json`, `.env` | credentials — committing these publishes your keys |
| Large data files | repositories aren't file storage |

The difference between `.venv/` and `requirements.txt` is the whole idea
of this lab: don't ship the environment, ship the **recipe** for
building it.

### Exercise 9.1
Write a `.gitignore` covering everything above.

In [ ]:
%%writefile .gitignore
# TODO: add the patterns


In [ ]:
ignored = open(".gitignore").read()

for pattern in [".venv/", "__pycache__/", "logs/", "kaggle.json"]:
    assert pattern in ignored, f"{pattern} is missing from .gitignore"
print("Exercise 9.1 correct.")

> **If you already committed something by mistake**, adding it to
> `.gitignore` afterwards does *not* remove it — Git keeps tracking
> files it already knows about. Use
> `git rm --cached <file>` to stop tracking it, then commit. And if it
> was a credential, treat it as compromised and regenerate it.

---
# Part 10 · Automated tests

You've been checking your work by reading printed output. That doesn't
scale, and it doesn't protect you: change one function, and you have no
idea whether you broke something written three weeks ago.

A **test** is code that checks your code, so you can re-verify
everything in a second.

### Writing tests

A test file lives in `tests/`, is named `test_*.py`, and contains
functions named `test_*`. Each uses `assert` to state what should be
true.

In [ ]:
%%writefile tests/test_grade_utils.py
"""Tests for src/grade_utils.py"""
import sys
import os

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "src"))

import grade_utils


def test_letter_grade_boundaries():
    """Each boundary value should fall into the higher grade."""
    assert grade_utils.letter_grade(4.50) == "A"
    assert grade_utils.letter_grade(3.50) == "B"
    assert grade_utils.letter_grade(2.50) == "C"
    assert grade_utils.letter_grade(1.50) == "D"


def test_letter_grade_typical():
    assert grade_utils.letter_grade(5.00) == "A"
    assert grade_utils.letter_grade(3.90) == "B"
    assert grade_utils.letter_grade(0.00) == "F"


def test_dean_list():
    assert grade_utils.is_dean_list(4.50) is True
    assert grade_utils.is_dean_list(4.49) is False


Notice the tests check **boundaries** — 4.50 and 4.49, the exact values
where behaviour changes. That's where bugs live. A test that only checks
`letter_grade(5.0) == "A"` would pass even if the threshold were wrong.

### Running them

In the terminal:

```bash
python -m pip install pytest
python -m pytest tests/ -v
```

Let's run it from here.

In [ ]:
import subprocess

outcome = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v", "--no-header"],
    capture_output=True, text=True,
)
print(outcome.stdout[-1500:])

### Exercise 10.1
Write `tests/test_text_utils.py` with **two** test functions for
`clean_name` from Part 4:

- `test_clean_name_whitespace` — collapses inner and outer whitespace
- `test_clean_name_capitalisation` — fixes all-caps and all-lowercase

Copy the two `sys.path` lines from the file above so the test can find
`src/`.

In [ ]:
%%writefile tests/test_text_utils.py
"""Tests for src/text_utils.py"""
import sys
import os

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "src"))

import text_utils


def test_clean_name_whitespace():
    # TODO
    pass


def test_clean_name_capitalisation():
    # TODO
    pass


In [ ]:
outcome = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v", "--no-header"],
    capture_output=True, text=True,
)
print(outcome.stdout[-2000:])

# A test function containing only "pass" passes without checking anything,
# so confirm each of yours actually asserts something.
test_source = open("tests/test_text_utils.py").read()

assert outcome.returncode == 0, "some tests failed - read the output above"
assert "test_clean_name_whitespace" in outcome.stdout, "test_clean_name_whitespace did not run"
assert "test_clean_name_capitalisation" in outcome.stdout, "test_clean_name_capitalisation did not run"
assert test_source.count("assert") >= 2, \
    "each of your two test functions needs at least one assert - an empty test always passes"
assert "pass" not in test_source, "remove the placeholder 'pass' lines once you've written real assertions"
print()
print("Exercise 10.1 correct.")

---
# Part 11 · The reproducibility test

Everything so far has been preparation for one question:

**Does your project run on a machine that has never seen it?**

Here's how to find out. **All of this runs in the terminal.**

### Step 1 — record your dependencies

```bash
python -m pip freeze > requirements.txt
```

Open the file and check it's pinned with `==`.

### Step 2 — commit

```bash
git add .
git commit -m "lab03: project structure, error handling, logging, tests"
git push
```

Then look at your repository on GitHub and confirm `.venv/` and `logs/`
are **not** there.

### Step 3 — destroy your environment

This is the part that feels wrong. Do it anyway — that's the test.

```bash
deactivate
rm -rf .venv          # Windows PowerShell: Remove-Item -Recurse -Force .venv
```

### Step 4 — rebuild from the recipe alone

```bash
python -m venv .venv
source .venv/bin/activate      # Windows: .venv\Scripts\Activate.ps1
python -m pip install -r requirements.txt
python -m pytest tests/ -v
```

**If your tests pass, your project is reproducible.** If they don't,
something was living on your machine that never made it into the
recipe — which is exactly the failure this lab exists to prevent. Fix
it and run the test again.

### Step 5 — swap with a classmate

Send them your repository link and clone theirs:

```bash
git clone <their-repo-url>
cd <their-repo>
python -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
python -m pytest tests/ -v
```

Then tell each other what broke. Your own README always looks complete
to you, because you already know the missing step. Someone else's
machine is the only honest reviewer you have.

---
# Before you finish

- [ ] Every self-check cell prints "correct"
- [ ] `src/`, `tests/`, `data/`, `logs/` exist, with your modules in `src/`
- [ ] `requirements.txt` is pinned with `==`
- [ ] `.gitignore` covers `.venv/`, `__pycache__/`, `logs/`, and credentials
- [ ] `python -m pytest tests/ -v` passes
- [ ] The reproducibility test in Part 11 passes from a rebuilt environment
- [ ] **Restart Kernel and Run All** — the whole notebook runs with no errors
- [ ] Committed and pushed